# 2.5 — does focal loss add anything on top of rotation?

Five runs, all on `baseline_v2` with free-angle rotation. Only the imbalance handling varies,
so any difference is attributable to it.

| config | loss | gamma | sampler |
|---|---|---|---|
| `control` | cross-entropy | — | inverse-sqrt |
| `focal_g2` | focal | 2.0 | none |
| `focal_g1` | focal | 1.0 | none |
| `focal_g2_sampler` | focal | 2.0 | inverse-sqrt |
| `focal_g1_sampler` | focal | 1.0 | inverse-sqrt |

**One correction before starting.** Focal loss has *not* been shown to beat the baseline. In
the phase-2 sweep it scored 0.8246 against the baseline's 0.8173 — a gap of 0.007 against a
noise floor of roughly 0.013, so the two are indistinguishable. Rotation is the only
intervention that cleared its interval, at +0.063. This series is worth running because
rotation changed the regime, not because focal already looked promising.

**The mechanism to watch.** The sampler makes rare classes common within a batch; focal then
down-weights examples once the model classifies them correctly. So focal's suppression falls
hardest on exactly what the sampler was promoting — the two can partly cancel rather than
compound. That is the hypothesis the `*_sampler` arms test, and the reason lower gamma is
the interesting direction rather than higher.

**How to judge the result.** Compare against `control`, never against the old `baseline_cnn`
numbers — both the model and the augmentation have changed since. And treat a small win with
suspicion: we are deep enough into validation selection that roughly +0.02 of optimism is
already priced in, so anything under that needs the frozen test split before it counts.

## 1. Setup

Same bootstrap as the phase-2 notebook; every step is a no-op if already done.

In [ ]:
import os, shutil, subprocess, sys
from pathlib import Path


def looks_like_the_repository(path: Path) -> bool:
    return (path / "pyproject.toml").exists() and (path / "src" / "fdl_project").is_dir()


REPO = next(
    (p for p in [Path.cwd(), *Path.cwd().parents, Path("/content/fdl-project")]
     if looks_like_the_repository(p)),
    None,
)
assert REPO is not None, "clone the repo to /content/fdl-project first"
os.chdir(REPO)

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--ignore-requires-python",
                "-e", str(REPO), "--no-deps"], check=True)
source = str(REPO / "src")
if source not in sys.path:
    sys.path.insert(0, source)

import torch

DRIVE_ROOT = Path("/content/drive/MyDrive")
DRIVE = DRIVE_ROOT / "BICOCCA/FDL"
DATASET = REPO / "data/MIR-WM811K/WM811K.pkl"
EXPECTED_BYTES = 2_022_961_642


def mount_drive() -> bool:
    if DRIVE_ROOT.is_dir():
        return True
    try:
        from google.colab import drive

        drive.mount("/content/drive")   # idempotent; never force_remount
    except Exception as error:
        print(f"  Drive unavailable ({type(error).__name__})")
        return False
    return DRIVE_ROOT.is_dir()


HAS_DRIVE = mount_drive()
if not DATASET.exists():
    DATASET.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(DRIVE / "DATA/data/MIR-WM811K/WM811K.pkl", DATASET)
assert DATASET.stat().st_size == EXPECTED_BYTES, "wrong pickle: splits are row indices"

CHECKPOINTS = DRIVE / "checkpoints"
if HAS_DRIVE:
    CHECKPOINTS.mkdir(parents=True, exist_ok=True)

print(f"gpu     {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE'}")
print(f"drive   {'mounted' if HAS_DRIVE else 'NOT mounted'}")
print(f"dataset {DATASET.stat().st_size / 1024**3:.2f} GiB")

## 2. W&B

`wandb login` in the terminal is the reliable path — Colab Secrets time out when the runtime
is driven from VS Code. `~/.netrc` is read by both this kernel and any terminal process.

In [ ]:
USE_WANDB = True
WANDB_PROJECT = "wm811k-wafer-defects"

if USE_WANDB:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "wandb"], check=True)
    import wandb

    if not wandb.api.api_key:
        USE_WANDB = False
        print("  not authenticated -- run `wandb login` in the terminal, then rerun")
    else:
        print(f"  wandb ready, project {WANDB_PROJECT!r}")

## 3. Run the five arms

Sequentially, on one loaded copy of the source table. `transform_device=cuda` is on: rotation
becomes a single batched `grid_sample` and the encoding happens after the transfer, which is
worth the most on precisely this augmentation.

Each arm writes its own artifacts and appends to the results list as it finishes, so an
interrupted session keeps whatever completed.

In [ ]:
import time

import pandas as pd

from fdl_project.config.loader import load_experiment_config
from fdl_project.data.datasets import load_wm811k_dataframe
from fdl_project.training.runner import run_experiment

CONFIGS = sorted((REPO / "configs/train/v25_focal").glob("*.yaml"))
OVERRIDES = ["data.transform_device=cuda"]
if HAS_DRIVE:
    OVERRIDES.append(f"checkpoint.directory={CHECKPOINTS}")
if USE_WANDB:
    OVERRIDES += ["logging.wandb.enabled=true",
                  f"logging.wandb.project={WANDB_PROJECT}"]

dataframe = load_wm811k_dataframe(DATASET)
results = []

for path in CONFIGS:
    config = load_experiment_config(path, overrides=OVERRIDES)
    print(f"\n=== {config.name}  ({config.imbalance.loss}, gamma "
          f"{config.imbalance.focal_gamma}, sampling {config.imbalance.sampling})")
    started = time.monotonic()
    result = run_experiment(config, overwrite=True, dataframe=dataframe)
    macro = result.bootstrap.aggregate.set_index("metric").loc["macro_f1"]
    results.append({
        "arm": path.stem,
        "loss": config.imbalance.loss,
        "gamma": config.imbalance.focal_gamma if config.imbalance.loss == "focal" else None,
        "sampler": config.imbalance.sampling,
        "macro_f1": round(float(macro.point_estimate), 4),
        "ci_lower": round(float(macro.ci_lower), 4),
        "ci_upper": round(float(macro.ci_upper), 4),
        "best_epoch": result.fit.best_epoch,
        "epochs": len(result.fit.history),
        "minutes": round((time.monotonic() - started) / 60, 1),
    })
    print(f"    macro-F1 {macro.point_estimate:.4f} "
          f"[{macro.ci_lower:.4f}, {macro.ci_upper:.4f}]  "
          f"{results[-1]['minutes']:.1f} min")

## 4. Read the result

`clears_control` is the test that matters: an arm counts only if its interval sits entirely
above the control's point estimate. With nine classes and 30 `Near-full` wafers in
validation, point estimates alone are not a ranking.

In [ ]:
frame = pd.DataFrame(results)
control = frame.loc[frame["arm"] == "control", "macro_f1"].squeeze()
frame["vs_control"] = (frame["macro_f1"] - control).round(4)
frame["clears_control"] = frame["ci_lower"] > control

pd.set_option("display.width", 200)
display(frame.sort_values("macro_f1", ascending=False))

winners = frame[frame["clears_control"] & (frame["arm"] != "control")]
if winners.empty:
    print("\nNothing clears the control. Imbalance handling is not the binding "
          "constraint here either -- report that as the finding.")
else:
    print("\nClears the control:", ", ".join(winners["arm"]))
    print("Treat gains under ~0.02 as unconfirmed until the test split is opened.")

In [ ]:
OUTPUT = REPO / "output/v25_focal"
OUTPUT.mkdir(parents=True, exist_ok=True)
frame.to_csv(OUTPUT / "results.csv", index=False)
if HAS_DRIVE:
    # Local disk dies with the session; Drive does not.
    shutil.copy2(OUTPUT / "results.csv", DRIVE / "v25_focal_results.csv")
    print("copied to", DRIVE / "v25_focal_results.csv")
print(OUTPUT / "results.csv")

## 5. What follows

If an arm clears the control, it becomes part of the pipeline and the next series runs on
top of it. If nothing does — the likelier outcome given that every imbalance arm in phase 2
landed inside ±0.013 — then the finding is that **augmentation is the only lever that
mattered on this dataset**, and the report says so with the numbers behind it.

Either way the next step is the same: three seeds on whichever setup wins, to separate a
real difference from selection on a noisy validation metric.